# Linear Regression on the Automobile Dataset

## Problem formulation

**Goal:** Predict a vehicle's horsepower from its other numerical characteristics.

- **Target (y):** `horsepower`
- **Features (X):** available numerical vehicle attributes, excluding `horsepower`
- **Model:** scikit-learn `LinearRegression`
- **Evaluation:** MAE, RMSE and R²

This follows the structure of the reference notebook, adapted to this dataset and target.


## Imports

In [2]:
from pathlib import Path
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


## Load and explore the data

In [ ]:
DATA_PATH = Path("data/Automobile.csv")
TARGET = "horsepower"

def normalize_column(name):
    return re.sub(r"[^a-z0-9]+", "-", str(name).strip().lower()).strip("-")

if not DATA_PATH.exists():
    raise FileNotFoundError("Download the Kaggle CSV and place it at data/Automobile_data.csv")

df = pd.read_csv(DATA_PATH, na_values=["?", "NA", "N/A", ""])
df.columns = [normalize_column(c) for c in df.columns]

print(f"Shape: {df.shape}")
display(df.head())


Shape: (398, 9)


,name,mpg,cylinder-,di-placement,hor-epower,weight,acceleration,model-year,origin
0,chevrolet chevelle malibu,18.0,8,307.0,130.0,3504,12.0,70,usa
1,buick skylark 320,15.0,8,350.0,165.0,3693,11.5,70,usa
2,plymouth satellite,18.0,8,318.0,150.0,3436,11.0,70,usa
3,amc rebel sst,16.0,8,304.0,150.0,3433,12.0,70,usa
4,ford torino,17.0,8,302.0,140.0,3449,10.5,70,usa


In [5]:
df.info()
print("\\nMissing values:")
display(df.isnull().sum())


<class 'pandas.DataFrame'>
RangeIndex: 398 entries, 0 to 397
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   name          398 non-null    str    
 1   mpg           398 non-null    float64
 2   cylinder-     398 non-null    int64  
 3   di-placement  398 non-null    float64
 4   hor-epower    392 non-null    float64
 5   weight        398 non-null    int64  
 6   acceleration  398 non-null    float64
 7   model-year    398 non-null    int64  
 8   origin        398 non-null    str    
dtypes: float64(4), int64(3), str(2)
memory usage: 28.1 KB
\nMissing values:


name            0
mpg             0
cylinder-       0
di-placement    0
hor-epower      6
weight          0
acceleration    0
model-year      0
origin          0
dtype: int64

## Preprocessing

`horsepower` is the target. It must not also be included among the input features.
The code selects the listed numeric features that exist in the downloaded CSV,
converts malformed values and `?` markers to missing values, and removes rows
with missing values in the selected columns.


In [ ]:
candidate_features = [
    "displacement",
    "weight",
    "acceleration",
    "cylinders",
    "mpg",
    "model-year"
]
feature_cols = [c for c in candidate_features if c in df.columns]

if TARGET not in df.columns:
    raise ValueError(f"Target '{TARGET}' not found. Columns: {df.columns.tolist()}")
if not feature_cols:
    raise ValueError(f"No expected feature columns found. Columns: {df.columns.tolist()}")

for column in [TARGET, *feature_cols]:
    df[column] = pd.to_numeric(df[column], errors="coerce")

df = df[[*feature_cols, TARGET]].dropna().copy()
print(f"Rows after cleaning: {len(df)}")
print(f"Features used: {feature_cols}")
display(df.describe())


ValueError: Target 'horsepower' not found. Columns: ['name', 'mpg', 'cylinder-', 'di-placement', 'hor-epower', 'weight', 'acceleration', 'model-year', 'origin']

## Train/test split and scaling

In [ ]:
X = df[feature_cols]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Train size: {len(X_train)}, Test size: {len(X_test)}")


## Baseline — Linear Regression

In [ ]:
lr = LinearRegression()
lr.fit(X_train_scaled, y_train)
y_pred = lr.predict(X_test_scaled)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("=== Linear Regression ===")
print(f"MAE:  {mae:.2f} horsepower")
print(f"RMSE: {rmse:.2f} horsepower")
print(f"R²:   {r2:.3f}")


## Feature coefficients

In [ ]:
coef_df = pd.DataFrame({
    "feature": feature_cols,
    "coefficient": lr.coef_,
}).sort_values("coefficient", key=np.abs, ascending=False)

display(coef_df)


## Actual vs. predicted horsepower

In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(y_test, y_pred, alpha=0.75)
low = min(float(y_test.min()), float(y_pred.min()))
high = max(float(y_test.max()), float(y_pred.max()))
plt.plot([low, high], [low, high], linestyle="--")
plt.xlabel("Actual horsepower")
plt.ylabel("Predicted horsepower")
plt.title("Linear Regression: actual vs. predicted horsepower")
plt.tight_layout()
plt.show()


## Export the fixed split for the LLM + RAG comparison

Only `train_reference.csv` may be added to the RAG knowledge base. Keep
`test_holdout.csv` out of the RAG collection and do not expose its horsepower
values to the LLM. Use the held-out targets only after both methods have predicted.


In [ ]:
train_reference = X_train.copy()
train_reference[TARGET] = y_train
train_reference.to_csv("data/train_reference.csv", index=False)

test_holdout = X_test.copy()
test_holdout[TARGET] = y_test
test_holdout.to_csv("data/test_holdout.csv", index=False)

predictions = pd.DataFrame({
    "actual_horsepower": y_test,
    "linear_regression_prediction": y_pred,
    "absolute_error": np.abs(y_test.to_numpy() - y_pred),
}, index=y_test.index)
predictions.to_csv("data/linear_regression_test_predictions.csv")

display(predictions.head(10).round(2))
